# Normal Reviews: add bars and pubs for CA, OR, WA, NV

The first download missed the names of bars, pubs and lounges. This gets just those.

**Click Runtime → Run all.** About 15–30 minutes (the second box is the slow one). One file downloads at the end: **west_bars.tsv.gz**. Attach it to the chat with Claude.
If anything turns red, copy the error text and paste it to Claude.

In [ ]:
!pip -q install --upgrade duckdb
import duckdb, pandas as pd, re, time
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs; SET s3_region='us-west-2';")
paths = con.sql("SELECT file FROM glob('s3://overturemaps-us-west-2/release/*/theme=places/type=place/*.parquet')").df()["file"]
rel = sorted({re.search(r"release/([^/]+)/", p).group(1) for p in paths})[-1]
SRC = f"s3://overturemaps-us-west-2/release/{rel}/theme=places/type=place/*.parquet"
cols = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{SRC}')").df()["column_name"].tolist()
print("Overture release:", rel)
print("Columns:", cols)

In [ ]:
# Category: a detailed one (e.g. mexican_restaurant) plus a basic one (e.g. restaurant)
if "taxonomy" in cols:     DETAIL = "taxonomy.primary"
elif "categories" in cols: DETAIL = "categories.primary"
else:                      DETAIL = "basic_category"
BASIC = "basic_category" if "basic_category" in cols else DETAIL
CONF  = "confidence" if "confidence" in cols else "NULL"
CITY  = "addresses[1].locality" if "addresses" in cols else "NULL"
STATE = "addresses[1].region" if "addresses" in cols else "NULL"
OPEN  = "operating_status" if "operating_status" in cols else "NULL"
# Box covering California, Oregon, Washington and Nevada
q = f"""SELECT names.primary AS name, {BASIC} AS basic, {DETAIL} AS detail, {CONF} AS conf,
       (bbox.ymin + bbox.ymax) / 2 AS lat, (bbox.xmin + bbox.xmax) / 2 AS lon,
       {CITY} AS city, {STATE} AS state, {OPEN} AS status
FROM read_parquet('{SRC}')
WHERE bbox.xmin > -124.9 AND bbox.xmax < -114.0 AND bbox.ymin > 32.4 AND bbox.ymax < 49.1"""
t = time.time(); print("Downloading places for CA, OR, WA, NV… (15–30 minutes)")
df = con.sql(q).df()
print(len(df), "places in", round((time.time() - t) / 60, 1), "minutes")

In [ ]:
from google.colab import files
BARS = r"(^|[_ ])(bar|pub|lounge|tavern|saloon|taproom|speakeasy|izakaya|nightlife_venue|cocktail_bar|sports_bar|dive_bar|gastropub|brewpub|beer_bar|hookah_bar|karaoke_bar|piano_bar|gay_bar|irish_pub|food_service|food_and_drink)($|[_ ])"
NOT = (r"public_|bed_and_breakfast|grocery|supermarket|convenience|liquor|butcher|food_bank|delivery|wholesale|catering|equipment|supply|"
       r"distributor|bartender|barber|bar_association|beauty_bar|nail_bar|oxygen_bar|lash_bar|brow_bar|blow_dry|vending|law|attorney")
d = df.copy()
d = d[d["conf"].isna() | (d["conf"] >= 0.4)]
d = d[~d["status"].astype(str).str.contains("closed", case=False, na=False)]
d["basic"] = d["basic"].astype(str); d["detail"] = d["detail"].fillna(d["basic"]).astype(str)
key = d["detail"] + " " + d["basic"]
pick = key.str.contains(BARS, regex=True) & ~key.str.contains(NOT, regex=True) & d["name"].notna()
bars = d[pick][["lat","lon","name","basic","detail","city","state"]].copy()
bars["lat"] = bars["lat"].round(6); bars["lon"] = bars["lon"].round(6)
bars["name"] = bars["name"].astype(str).str.replace("\t", " ").str.replace("\n", " ")
bars = bars.fillna("").drop_duplicates().sort_values(["lat","lon"])
print(len(bars), "bars, pubs and lounges"); print(bars["detail"].value_counts().head(15))
bars.to_csv("west_bars.tsv.gz", sep="\t", index=False, compression="gzip")
files.download("west_bars.tsv.gz")
